## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score)
from sklearn.feature_selection import SelectKBest, f_classif, RFE

df = pd.read_csv('../data/df_model.csv', sep=';', dtype={'CODE_INSEE': str})
display(df.head(5))
print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

print(f"\nDistribution de la cible (vote politique) :")
print(df['RESULT'].value_counts())
print(f"\nProportion :")
print(df['RESULT'].value_counts(normalize=True))

## Split test-train 30-70

In [ ]:
X = df.drop(['RESULT', 'CODE_INSEE', 'CITY'], axis=1)
y = df['RESULT']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("✓ Données préparées")
print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")

## Pipeline basique - comparaison de modèles

In [ ]:
# Fonction pour entraînement et prédiction
def train_and_predict(pipeline):
    pipeline.fit(X_train, y_train)
    print(f"✓ Pipeline entraîné !")


    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Précision : {accuracy:.2%}")


# Fonction pour détecter l'overfitting
def detect_overfitting(pipeline):
    pipeline.fit(X_train, y_train)

    train_score = pipeline.score(X_train, y_train)
    test_score = pipeline.score(X_test, y_test)
    gap = train_score - test_score


    print(f"  Train : {train_score:.2%}")
    print(f"  Test  : {test_score:.2%}")
    print(f"  Gap   : {gap:.2%} ", end="")


    if gap > 0.10:
        print("⚠️ OVERFITTING détecté !")
    elif test_score < 0.75:
        print("⚠️ UNDERFITTING détecté !")
    else:
        print("✅ Bon équilibre")


### Random Forest

In [ ]:
pipeline_forest = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])


detect_overfitting(pipeline_forest)


### Régression logistique

In [ ]:
pipeline_logistic = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42))
])


detect_overfitting(pipeline_logistic)

### Arbre de décision

In [ ]:
pipeline_tree = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])


detect_overfitting(pipeline_tree)

## Observations

__Random Forest et Decision Tree ont 100 % de précision sur le train.
Les paramètres n'étant pas ajustés, ils arrivent à identifier une commune d'après le nombre d'habitants, etc.__

Solutions :
- Ajustement des paramètres

OU

- Transformer le nombre d'habitants en variable catégorielle (village, ville moyenne, grande ville, ville principale, etc.) + transformer le nombre d'inscrits, etc. en pourcentage par rapport à la population totale


**Pour l'optimisation des modèles :** Possibilité de s'y mettre à 3, chacun essaye d'optimiser un modèle.

## Optimisation basique de Logistic Regression

### Validation croisée

In [ ]:
# Fonction pour la validation croisée
def get_cross_val_scores(pipeline):
    cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

    print("Scores de validation croisée (5-fold) :")
    print(f"  Scores : {[f'{score:.2%}' for score in cv_scores]}")
    print(f"  Moyenne : {cv_scores.mean():.2%} (+/- {cv_scores.std():.2%})")

get_cross_val_scores(pipeline_logistic)

### Sélection de features

In [ ]:
pipeline_selection = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=5)),
    ('classifier', LogisticRegression(random_state=42))
])

train_and_predict(pipeline_selection)


selector = pipeline_selection.named_steps['selector']
features_selected = X.columns[selector.get_support()].tolist()
print("\nFeatures sélectionnées :")
for i, feature in enumerate(features_selected):
    print(f"   {i+1}- {feature}")

In [ ]:
# Comparaison avec différents nombres de features
k_values = [5, 10, 15, 20, 25, 30, 35, 40]
results = []

for k in k_values:
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(f_classif, k=k)),
        ('classifier', RandomForestClassifier(random_state=42))
    ])
    
    cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
    results.append({
        'k': k,
        'mean_accuracy': cv_scores.mean(),
        'std_accuracy': cv_scores.std()
    })

# Affichage des résultats
results_df = pd.DataFrame(results)
print("Résultats selon le nombre de features :")
print(results_df.to_string(index=False))